## Sel 1: Import Pustaka

In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

import mlflow
import dagshub

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import xgboost as xgb

import optuna

import joblib
import contextlib
from tqdm.auto import tqdm

import warnings
warnings.filterwarnings('ignore')

## Sel 2: Memuat Data (Resampling Harian)

In [ ]:
# Pastikan path ini sesuai dengan posisi .env milikmu
load_dotenv('../.env', override=True)

# Ambil URL MLflow utuh 
db_url_mlflow = os.getenv("DATABASE_URL_DIRECT")

if not db_url_mlflow:
    raise ValueError("DATABASE_URL_DIRECT tidak ditemukan di .env!")

# Buat URL polos khusus untuk Pandas & SQLAlchemy
db_url_pandas = db_url_mlflow.split("?")[0]

print("Membuka jembatan ke Supabase...")
# Gunakan URL polos untuk engine
engine = create_engine(db_url_pandas)

# ==========================================
# LANGKAH 1: KUERI SQL JATAH RATA
# ==========================================
RENTANG_WAKTU = "1 year"

query_sql = f"""
    SELECT 
        waktu_aktual, 
        id_wilayah, 
        pm25, pm10, so2, co, no2, ozon
    FROM public.data_historis
    WHERE waktu_aktual >= NOW() - INTERVAL '{RENTANG_WAKTU}';
"""

ukuran_chunk = 50000
chunks = []

with engine.connect() as conn:
    conn.execute(text("SET statement_timeout = 0"))
    conn.commit() 
    
    stream_conn = conn.execution_options(stream_results=True)
    
    for i, chunk in enumerate(pd.read_sql(text(query_sql), stream_conn, chunksize=ukuran_chunk)):
        print(f"-> Berhasil menyedot batch ke-{i+1} (hingga {(i+1) * ukuran_chunk} baris...)")
        chunks.append(chunk)

df_raw = pd.concat(chunks, ignore_index=True)
print(f"\n[+] Total data mentah ditarik: {df_raw.shape}")

## Sel 3: Rekayasa Fitur

In [ ]:
# =====================================================================
# 3. PREPROCESSING & REKAYASA FITUR (DATA ENGINEERING)
# =====================================================================
print("[~] Memulai proses pembersihan dan rekayasa fitur...")
df_clean = df_raw.copy()

kolom_waktu = 'waktu_aktual'
kolom_kota = 'id_wilayah'  
polutan = ['pm25', 'pm10', 'so2', 'co', 'no2', 'ozon'] 

# ---------------------------------------------------------
# A. PRA-PEMROSESAN & RESAMPLE 
# ---------------------------------------------------------
df_clean[kolom_waktu] = pd.to_datetime(df_clean[kolom_waktu])
df_clean[kolom_waktu] = df_clean[kolom_waktu].dt.tz_localize(None)
df_clean.set_index(kolom_waktu, inplace=True)

df_hourly = df_clean.groupby(kolom_kota)[polutan].resample('H').mean().reset_index()

# ---------------------------------------------------------
# B. PEMBERSIHAN OUTLIER (Sensor Error) -> SEBELUM INTERPOLASI
# ---------------------------------------------------------
# Ubah nilai mustahil (<= 0) menjadi NaN agar ikut di-interpolasi
for col in polutan:
    df_hourly.loc[df_hourly[col] <= 0, col] = np.nan

# ---------------------------------------------------------
# C. INTERPOLASI & WINSORIZING
# ---------------------------------------------------------
print("Menambal data sensor yang bolong dengan interpolasi...")
df_hourly = df_hourly.groupby(kolom_kota, group_keys=False).apply(
    lambda group: group.interpolate(method='linear')
).reset_index(drop=True)

df_hourly = df_hourly.bfill().ffill()

# Potong lonjakan ekstrem (Winsorizing 99.9%)
for col in polutan:
    batas_atas = df_hourly[col].quantile(0.999)
    df_hourly[col] = np.where(df_hourly[col] > batas_atas, batas_atas, df_hourly[col])

# ---------------------------------------------------------
# D. PEMBUATAN FITUR TEMPORAL (LAG & ROLLING)
# ---------------------------------------------------------
print("Mengekstrak fitur temporal, Lag, dan Rolling Mean...")
df_hourly['jam'] = df_hourly[kolom_waktu].dt.hour
df_hourly['hari_dalam_minggu'] = df_hourly[kolom_waktu].dt.dayofweek
df_hourly['bulan'] = df_hourly[kolom_waktu].dt.month

fitur_baru = {}
for p in polutan:
    for lag in range(1, 25):
        fitur_baru[f'{p}_H-{lag}'] = df_hourly.groupby(kolom_kota)[p].shift(lag)
    
    fitur_baru[f'{p}_RollMean_72'] = df_hourly.groupby(kolom_kota)[p].transform(
        lambda x: x.rolling(window=72, min_periods=1).mean()
    )
    fitur_baru[f'{p}_RollMax_72'] = df_hourly.groupby(kolom_kota)[p].transform(
        lambda x: x.rolling(window=72, min_periods=1).max()
    )

df_features = pd.concat([df_hourly, pd.DataFrame(fitur_baru)], axis=1)

# ---------------------------------------------------------
# E. TARGET PREDIKSI (T+1 s/d T+24)
# ---------------------------------------------------------
print("Menciptakan target prediksi...")
target_cols = {}
for p in polutan:
    for t in range(1, 25):
        target_cols[f'target_{p}_{t}j'] = df_features.groupby(kolom_kota)[p].shift(-t)

df_model = pd.concat([df_features, pd.DataFrame(target_cols)], axis=1)

# ---------------------------------------------------------
# F. ONE-HOT ENCODING & FINALISASI
# ---------------------------------------------------------
df_model = pd.get_dummies(df_model, columns=[kolom_kota, 'jam', 'hari_dalam_minggu', 'bulan'], drop_first=True)

# Hapus baris NaN akibat fungsi Shift
df_model.dropna(inplace=True)
df_model.drop(columns=[kolom_waktu], inplace=True)

print(f"\n[+] Rekayasa fitur selesai! Bentuk data akhir: {df_model.shape}")

# =========================================================
# MENAMPILKAN DAFTAR KOLOM
# =========================================================
daftar_kolom = df_model.columns.tolist()
print("\n[+] Informasi Kolom:")
print(f"Total jumlah kolom: {len(daftar_kolom)}")
# print("\n--- 20 Kolom Pertama (Biasanya Polutan Aktual & Lag) ---")
# print(daftar_kolom[:20])

# print("\n--- 20 Kolom Terakhir (Biasanya Hasil One-Hot Encoding) ---")
# print(daftar_kolom[-20:])

# Jika butuh melihat semuanya (bisa sangat panjang), aktifkan baris di bawah ini:
print("\n--- Daftar Keseluruhan Kolom ---")
print(daftar_kolom)

# Menampilkan 5 baris teratas
display(df_model.head())

## Sel 4: Splitting & Tracking Baseline Model MLFLOW

### Splitting

In [ ]:
# =====================================================================
# 4. PEMISAHAN DATA (TRAIN-TEST SPLIT)
# =====================================================================

print("[~] Memisahkan Fitur (X) dan Target (y)...")

# 1. Definisikan X dan y secara dinamis
kolom_target = [col for col in df_model.columns if col.startswith('target_')]
kolom_fitur = [col for col in df_model.columns if not col.startswith('target_')]

X = df_model[kolom_fitur]
y = df_model[kolom_target]

# 2. Splitting Data (Time Series Split: Masa Lalu untuk Train, Masa Depan untuk Test)
print("[~] Membelah data menjadi set Pelatihan (80%) dan Pengujian (20%)...")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

# 3. Optimasi RAM (Opsional tapi sangat dianjurkan untuk dataset 2 tahun)
print("[~] Mengecilkan ukuran data di RAM (Downcasting ke Float32)...")
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')

print(f"[+] Splitting Selesai!")
print(f"    -> Dimensi X_train: {X_train.shape}")
print(f"    -> Dimensi X_test : {X_test.shape}")

# ==========================================
# TRANSFORMASI LOGARITMIK (ANTI-MINUS)
# ==========================================
print("Menerapkan Transformasi Logaritmik (Log1p) pada Target untuk mencegah prediksi negatif...")

# y = log(y + 1). Ini memaksa rentang data menjadi lebih rapat dan mencegah output negatif.
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

print("Data target telah dikompresi ke skala logaritmik!")
display(y_train_log.head(3))

### Tracking MLFlow DagsHub

In [ ]:
# ==========================================
# TRACKING MLFLOW DENGAN DAGSHUB
# ==========================================
# 2. Ambil Konfigurasi DagsHub dari .env
repo_owner = os.getenv("DAGSHUB_REPO_OWNER")
repo_name = os.getenv("DAGSHUB_REPO_NAME")

if not repo_owner or not repo_name:
    raise ValueError("Identitas DAGSHUB_REPO_OWNER atau DAGSHUB_REPO_NAME belum di-set di file .env!")

print(f"Connecting to DagsHub MLflow Server ({repo_owner}/{repo_name})...")
dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)

## Sel 5 : Hyperparameter Tuning XGBoost MLFlow

In [ ]:
print("Memulai Hyperparameter Tuning XGBoost Multioutput (Dengan Proteksi Logaritmik)...")

# 1. Set Eksperimen Tahap 2 (Nama Baru untuk Native Multi)
mlflow.set_experiment("XGB_Optuna_Native_MultiOutput")

print("\nMenyiapkan Data untuk Akselerasi GPU (Konversi ke Numpy C-Contiguous)...")
X_train_np = np.ascontiguousarray(X_train.values.astype('float32'))
X_test_np = np.ascontiguousarray(X_test.values.astype('float32'))

model_terbaik_per_polutan = {}
total_mae, total_rmse = 0, 0

# --- MENGELOMPOKKAN 144 TARGET MENJADI 6 GRUP ---
daftar_polutan = ['PM25', 'PM10', 'SO2', 'CO', 'NO2', 'OZON'] 
kelompok_target = {}

for polutan in daftar_polutan:
    kolom_terkait = [col for col in y_train_log.columns if polutan in col.upper()]
    if kolom_terkait:
        kelompok_target[polutan] = kolom_terkait
    else:
        print(f"PERINGATAN: Kolom untuk polutan {polutan} tidak ditemukan!")

# ==========================================
# 2. FUNGSI OPTUNA DENGAN TIME SERIES SPLIT
# ==========================================
def objective(trial, X_ctx, y_ctx):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 6), 
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 10.0, log=True)
    }
    
    model = xgb.XGBRegressor(
        **params, 
        random_state=42, 
        tree_method="hist", 
        device="cuda",
        multi_strategy="multi_output_tree",
        n_jobs=1
    )
    
    # Gunakan TimeSeriesSplit, BUKAN cv=3 biasa!
    tscv = TimeSeriesSplit(n_splits=3)
    score = cross_val_score(model, X_ctx, y_ctx, scoring='neg_mean_absolute_error', cv=tscv).mean()
    
    return score

# ==========================================
# 3. PROSES TUNING 6 POLUTAN (ZIP & UNZIP)
# ==========================================
for nama_polutan, list_kolom_24_jam in kelompok_target.items():
    print(f"\n======================================================")
    print(f"--> Mendidik AI Spesialis Khusus: {nama_polutan} (Native Multi-Output)")
    print(f"======================================================")
    
    y_train_grup_log = y_train_log[list_kolom_24_jam]
    y_train_grup_log_np = np.ascontiguousarray(y_train_grup_log.values.astype('float32'))
    
    y_test_grup_asli = y_test[list_kolom_24_jam]
    
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    study = optuna.create_study(direction='maximize')
    study.optimize(lambda trial: objective(trial, X_train_np, y_train_grup_log_np), n_trials=20, show_progress_bar=True)
    
    best_params = study.best_params
    print(f"[+] Kombinasi Terbaik Ditemukan:\n{best_params}")
    
    otak_spesialis = xgb.XGBRegressor(
        **best_params, 
        random_state=42, 
        tree_method="hist", 
        device="cuda",
        multi_strategy="multi_output_tree",
        n_jobs=1
    )
    
    otak_spesialis.fit(X_train_np, y_train_grup_log_np)
    model_terbaik_per_polutan[nama_polutan] = otak_spesialis
    
    # [FASE UNZIP]: Prediksi dan Kembalikan ke Wujud Asli (Microgram)
    y_pred_grup_log = otak_spesialis.predict(X_test_np)
    y_pred_grup_asli = np.expm1(y_pred_grup_log)
    
    # Menghitung MAE dan RMSE Keseluruhan
    mae_kombinasi = mean_absolute_error(y_test_grup_asli, y_pred_grup_asli)
    rmse_kombinasi = np.sqrt(mean_squared_error(y_test_grup_asli, y_pred_grup_asli))
    
    # --- IMPLEMENTASI R-SQUARED (R2) ---
    r2_kombinasi = r2_score(y_test_grup_asli, y_pred_grup_asli) # R2 Keseluruhan 24 jam
    r2_per_jam = r2_score(y_test_grup_asli, y_pred_grup_asli, multioutput='raw_values') # R2 spesifik tiap jam
    
    rata_rata_asli = y_test_grup_asli.values.mean()
    persentase_error = (mae_kombinasi / rata_rata_asli) * 100 if rata_rata_asli > 0 else 0
    
    total_mae += mae_kombinasi
    total_rmse += rmse_kombinasi
    
    print(f"  - MAE (Meleset Rata-rata) : {mae_kombinasi:.2f}")
    print(f"  - RMSE Rata-rata 24 Jam   : {rmse_kombinasi:.2f}")
    print(f"  - R-Squared (R2) Score    : {r2_kombinasi:.4f}") # Menampilkan skor R2 di terminal
    print(f"  - Error (%)               : {persentase_error:.2f}%")
    
    # PENCATATAN DAGSHUB
    with mlflow.start_run(run_name=f"Optuna_XGB_Native_{nama_polutan}_Log"):
        mlflow.log_params(best_params)
        mlflow.log_metric("test_mae_kombinasi", mae_kombinasi)
        mlflow.log_metric("test_rmse_kombinasi", rmse_kombinasi)
        mlflow.log_metric("test_r2_kombinasi", r2_kombinasi) # Mencatat R2 Keseluruhan
        mlflow.log_metric("error_percentage", persentase_error)
        
        # Mencatat R2 per jam ke MLflow (menggantikan RMSE per jam)
        for jam in range(len(list_kolom_24_jam)):
            mlflow.log_metric(f"r2_jam_ke_{jam+1}", r2_per_jam[jam])

print("\n--- 6 MODEL SELESAI DILATIH (NATIVE MULTI-OUTPUT + LOG TRANSFORM) ---")
if len(kelompok_target) > 0:
    print(f"Rata-rata MAE Keseluruhan ({len(kelompok_target)} Model) : {total_mae/len(kelompok_target):.2f}")

## Sel 6: Audit dan Evaluasi Performa Model dengan Visualisasi

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display

# Pastikan array menggunakan C-Contiguous untuk XGBoost
X_test_np = np.ascontiguousarray(X_test.values.astype('float32'))

# =====================================================================
# 1. TABEL STATISTIK AKURASI KESELURUHAN POLUTAN
# =====================================================================
print("\n[1] --- TABEL STATISTIK AKURASI TIAP POLUTAN (TEST SET) ---")
statistik_polutan = []

for polutan, model in model_terbaik_per_polutan.items():
    kolom_target_24jam = [col for col in y_test.columns if polutan in col.upper()]
    y_true_asli = y_test[kolom_target_24jam]
    
    # Prediksi dan kembalikan wujud logaritma ke asli (Microgram)
    y_pred_log = model.predict(X_test_np)
    y_pred_asli = np.expm1(y_pred_log)

    # Kalkulasi Metrik Global
    rata_rata_asli = y_true_asli.values.mean()
    mae = mean_absolute_error(y_true_asli, y_pred_asli)
    rmse = np.sqrt(mean_squared_error(y_true_asli, y_pred_asli))
    r2 = r2_score(y_true_asli, y_pred_asli)

    persentase_error = (mae / rata_rata_asli) * 100 if rata_rata_asli != 0 else 0

    statistik_polutan.append({
        "Polutan": polutan.upper(),
        "Rata-rata (µg/m³)": f"{rata_rata_asli:.2f}",
        "MAE": f"{mae:.2f}",
        "RMSE": f"{rmse:.2f}",
        "R-Squared (R²)": f"{r2:.4f}",
        "Error (%)": f"{persentase_error:.2f}%"
    })

df_statistik = pd.DataFrame(statistik_polutan)
display(df_statistik)

# =====================================================================
# 2. AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5)
# =====================================================================
print("\n[2] --- AUDIT KEADILAN AI PER WILAYAH (Khusus PM2.5 24-Jam) ---")
kunci_pm25 = 'PM25' 
if kunci_pm25 in model_terbaik_per_polutan:
    model_pm25 = model_terbaik_per_polutan[kunci_pm25]

    y_pred_log_pm25 = model_pm25.predict(X_test_np)
    y_pred_pm25 = np.expm1(y_pred_log_pm25) # Kembalikan ke asli
    
    kolom_pm25 = [col for col in y_test.columns if 'PM25' in col.upper()]
    y_test_pm25 = y_test[kolom_pm25]

    kolom_kota = 'id_wilayah'
    kolom_wilayah = [col for col in X_test.columns if col.startswith(f'{kolom_kota}_')]

    hasil_audit = []
    for wil in kolom_wilayah:
        mask = X_test[wil] == 1
        if mask.sum() > 0:
            y_test_wilayah = y_test_pm25[mask]
            y_pred_wilayah = y_pred_pm25[mask.values]

            mae_wilayah = mean_absolute_error(y_test_wilayah, y_pred_wilayah)
            id_asli = wil.replace(f'{kolom_kota}_', '')
            hasil_audit.append((int(id_asli), mae_wilayah))

    # Urutkan berdasarkan ID kota agar rapi
    hasil_audit.sort(key=lambda x: x[0])
    for id_kota, mae in hasil_audit:
        print(f"ID Wilayah {id_kota:<3} | MAE Prediksi PM2.5: {mae:.2f} µg/m³")
else:
    print("Model PM25 tidak ditemukan di memori!")

# =====================================================================
# 3. VISUALISASI UNTUK PPT 
# =====================================================================
print("\n[3] --- MEMBUAT DASHBOARD VISUALISASI UNTUK PRESENTASI ---")

# Kunci satu indeks acak agar sampel simulasi 24 jam konsisten untuk semua polutan
np.random.seed(42)
idx_acak = np.random.randint(0, X_test_np.shape[0])
jam_horizon = np.arange(1, 25)

# Siapkan daftar polutan yang berhasil dilatih
daftar_polutan = list(model_terbaik_per_polutan.keys())
jumlah_polutan = len(daftar_polutan)

sns.set_theme(style="whitegrid", context="talk")

# ---------------------------------------------------------------------
# SIMULASI PREDIKSI 24 JAM (Grid 2 Baris x 3 Kolom)
# ---------------------------------------------------------------------
fig1, axes1 = plt.subplots(2, 3, figsize=(24, 12))
fig1.suptitle('Dashboard Simulasi Prediksi 24 Jam (Aktual vs XGBoost)', fontsize=24, fontweight='bold', y=1.02)
axes1 = axes1.flatten() # Ratakan grid agar mudah di-loop

for i, polutan_vis in enumerate(daftar_polutan):
    if i >= 6: break # Maksimal 6 kotak
    
    model_vis = model_terbaik_per_polutan[polutan_vis]
    kolom_vis = [col for col in y_test.columns if polutan_vis.upper() in col.upper()]
    
    y_true_vis = y_test[kolom_vis].values
    y_pred_vis = np.expm1(model_vis.predict(X_test_np))

    axes1[i].plot(jam_horizon, y_true_vis[idx_acak], label='Aktual', marker='o', markersize=6, color='#1f77b4', lw=2)
    axes1[i].plot(jam_horizon, y_pred_vis[idx_acak], label='Prediksi', marker='X', markersize=6, linestyle='--', color='#d62728', lw=2)
    
    axes1[i].set_title(f'Polutan: {polutan_vis.upper()}', fontsize=16, fontweight='bold')
    axes1[i].set_xlabel('Jam Ke-', fontsize=12)
    axes1[i].set_ylabel('µg/m³', fontsize=12)
    axes1[i].set_xticks(jam_horizon[::4]) # Tampilkan label sumbu X tiap 4 jam agar tidak penuh
    axes1[i].legend(fontsize=10)

# Sembunyikan sisa kotak jika polutan kurang dari 6
for j in range(i + 1, 6):
    fig1.delaxes(axes1[j])

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------------
# KURVA DEGRADASI ERROR MAE (Grid 2 Baris x 3 Kolom)
# ---------------------------------------------------------------------
fig2, axes2 = plt.subplots(2, 3, figsize=(24, 12))
fig2.suptitle('Dashboard Kurva Degradasi Error Sepanjang Horizon Waktu', fontsize=24, fontweight='bold', y=1.02)
axes2 = axes2.flatten()

for i, polutan_vis in enumerate(daftar_polutan):
    if i >= 6: break
    
    model_vis = model_terbaik_per_polutan[polutan_vis]
    kolom_vis = [col for col in y_test.columns if polutan_vis.upper() in col.upper()]
    
    y_true_vis = y_test[kolom_vis].values
    y_pred_vis = np.expm1(model_vis.predict(X_test_np))

    # Hitung MAE per jam
    mae_per_jam = [mean_absolute_error(y_true_vis[:, h], y_pred_vis[:, h]) for h in range(24)]

    axes2[i].plot(jam_horizon, mae_per_jam, marker='s', color='#ff7f0e', lw=2, markersize=6)
    axes2[i].set_title(f'Error MAE: {polutan_vis.upper()}', fontsize=16, fontweight='bold')
    axes2[i].set_xlabel('Jam Ke-', fontsize=12)
    axes2[i].set_ylabel('Meleset (µg/m³)', fontsize=12)
    axes2[i].set_xticks(jam_horizon[::4])

# Sembunyikan sisa kotak jika polutan kurang dari 6
for j in range(i + 1, 6):
    fig2.delaxes(axes2[j])

plt.tight_layout()
plt.show()

print("\n[+] Dashboard Visualisasi siap untuk di-screenshot ke PPT!")

## Sel 7: Ekspor Model (Menyimpan Hasil)

In [ ]:
print("Memulai proses verifikasi dan ekspor model...")

# 1. SANITY CHECK: Pastikan ke-6 model polutan benar-benar ada di memori
daftar_polutan_wajib = ['PM25', 'PM10', 'SO2', 'CO', 'NO2', 'OZON']
polutan_hilang = [p for p in daftar_polutan_wajib if p not in model_terbaik_per_polutan]

if polutan_hilang:
    print(f"Batal Ekspor! PERINGATAN KRITIS: Otak model untuk {polutan_hilang} tidak ditemukan di memori!")
    
else:
    print("Pengecekan aman: 6 model spesialis lengkap di memori.")
    
    # 2. Simpan
    nama_file = "xgb_optuna_multioutput.pkl"

    # 3. Membungkus KE-ENAM model spesialis dan daftar urutan fitur
    paket_model = {
        'dict_model_spesialis': model_terbaik_per_polutan,
        'fitur': list(X_train.columns)
    }

    # 4. Simpan langsung ke kandang DVC
    joblib.dump(paket_model, nama_file)

    print(f"Model pintar telah diekspor sebagai '{nama_file}'")
    print(f"Total fitur cuaca/historis yang direkam: {len(paket_model['fitur'])} kolom")